In [1]:
# @title Step 1: Install Libraries
%pip install -qU langgraph langchain langchain-core langchain-google-genai PyGithub

In [2]:
# @title Step 2: Configure API Keys
import os
from google.colab import userdata

# Load keys from Colab secrets
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
os.environ['GITHUB_ACCESS_TOKEN'] = userdata.get('GITHUB_ACCESS_TOKEN')

print("API keys configured successfully!")

API keys configured successfully!


In [3]:
# @title Step 3: Define the Graph State
from typing import TypedDict, List, Dict

class GraphState(TypedDict):
  """
  Represents the state of our graph.

  Attributes:
      repo_url: The URL of the GitHub repository.
      user_query: The user's natural language query.
      raw_issues: The list of raw issue data fetched from Github.
      formatted_prompt: The final prompt sent to the AI.
      ai_response: The raw JSON string response from the AI.
      filtered_issues: The final, clean list of matching issue URLs.
      error: To store any error messages.
  """
  repo_url: str
  user_query: str
  raw_issues: List[Dict]
  formatted_prompt: str
  ai_response: str
  filtered_issues: List[str]
  error: str

In [4]:
# @title Step 4: Create Node 1 - Fetch GitHub Issues
from github import Github
import re

def fetch_github_issues_node(state: GraphState) -> GraphState:
    """Fetches open issues from a GitHub repository."""
    print("--- 1. FETCHING GITHUB ISSUES ---")
    try:
        g = Github(os.environ['GITHUB_ACCESS_TOKEN'])
        repo_url = state['repo_url']

        # Extract owner/repo from URL, just like in n8n
        match = re.search(r"github\.com/([^/]+)/([^/]+)", repo_url)
        if not match:
            return {"error": "Invalid GitHub repository URL."}

        owner, repo_name = match.groups()
        repo = g.get_repo(f"{owner}/{repo_name}")

        # Get the first 100 open issues (same as our final n8n setup)
        issues = repo.get_issues(state='open')

        raw_issues = []
        for issue in issues[:100]: # Limit to 100 to be efficient
            raw_issues.append({
                "id": issue.id,
                "title": issue.title,
                "body": issue.body,
                "html_url": issue.html_url,
                "labels": [label.name for label in issue.labels]
            })

        return {"raw_issues": raw_issues}
    except Exception as e:
        return {"error": f"Failed to fetch GitHub issues: {e}"}

In [5]:
# @title Step 5: Create Node 2 - Format for AI (Updated)
import json

def format_for_ai_node(state: GraphState) -> GraphState:
    """Prepares the data and prompt for the AI agent using the proven n8n prompt."""
    print("--- 2. FORMATTING DATA FOR AI ---")

    user_query = state['user_query']
    raw_issues = state['raw_issues']

    # Create the clean list of issues for the prompt
    formatted_issues_for_prompt = []
    for i, issue in enumerate(raw_issues):
      formatted_issues_for_prompt.append({
        "id": i,
        "url": issue['html_url'],
        "title": issue['title'],
        "body": issue['body'][:1500] if issue['body'] else "No body.", # Truncate body
        "labels": ", ".join(issue['labels'])
      })

    # This is your exact, working prompt from n8n
    system_message = f"""
You are an expert GitHub issue matching agent.
You will be given a user's request and a JSON array of GitHub issues. Your critical task is to analyze EVERY issue in the array and determine if it matches the user's request.
Your response MUST be a single, valid JSON array of objects. For each issue you analyzed, you must create a corresponding object in the array. Each object must contain the original "id" and a "url" key. The "url" key's value should be the issue's URL if it's a match, or the string "null" if it is not.

Do not provide any explanation or text outside of the final JSON array.

---
EXAMPLE INPUT:
USER REQUEST: "find me an easy issue about docs"
ISSUES:
[
  {{"id": 0, "title": "Fix typo in readme", "url": "http://a.com"}},
  {{"id": 1, "title": "Refactor database", "url": "http://b.com"}}
]

EXAMPLE OUTPUT:
[
  {{"id": 0, "url": "http://a.com"}},
  {{"id": 1, "url": "null"}}
]

DO NOT INCLUDE ANY MARKDOWN CODE DECORATORS like ```json give back the exact output
---

USER REQUEST:
{user_query}

ISSUES:
{json.dumps(formatted_issues_for_prompt, indent=2)}
"""
    return {"formatted_prompt": system_message}

In [6]:
# @title Step 6: Create Node 3 - Call AI (Gemini)
from langchain_google_genai import ChatGoogleGenerativeAI

def call_gemini_node(state: GraphState) -> GraphState:
    """Calls the Gemini model to get the analysis."""
    print("--- 3. CALLING GEMINI ---")

    # Initialize the Gemini 1.5 Pro model
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro", temperature=0)

    prompt = state['formatted_prompt']

    try:
        response = llm.invoke(prompt)
        return {"ai_response": response.content}
    except Exception as e:
        return {"error": f"Error calling Gemini: {e}"}

In [7]:
# @title Step 7: Create Node 4 - Parse and Filter Response (Updated)
def parse_and_filter_node(state: GraphState) -> GraphState:
    """Parses the AI's JSON response and filters out nulls."""
    print("--- 4. PARSING AND FILTERING RESPONSE ---")

    ai_response_text = state['ai_response']

    # Use regex to robustly find and clean the JSON blob
    json_match = re.search(r"```json\s*([\s\S]*?)\s*```|(\[[\s\S]*\])", ai_response_text)
    if not json_match:
        return {"error": "AI response did not contain a valid JSON array."}

    cleaned_text = json_match.group(1) or json_match.group(2)

    try:
        results = json.loads(cleaned_text)
        # --- THIS IS THE KEY CHANGE ---
        # We are now looking for the 'url' key instead of 'match'
        filtered_issues = [
            item['url'] for item in results if item['url'] != "null"
        ]
        return {"filtered_issues": filtered_issues}
    except json.JSONDecodeError as e:
        return {"error": f"Failed to parse AI JSON response: {e}", "ai_response": ai_response_text}

In [8]:
# @title Step 8: Assemble and Run the Graph
from langgraph.graph import StateGraph, END

# 1. Define the workflow
workflow = StateGraph(GraphState)

# 2. Add the nodes to the graph
workflow.add_node("fetch_issues", fetch_github_issues_node)
workflow.add_node("format_for_ai", format_for_ai_node)
workflow.add_node("call_gemini", call_gemini_node)
workflow.add_node("parse_and_filter", parse_and_filter_node)

# 3. Define the edges (the connections between nodes)
workflow.set_entry_point("fetch_issues")
workflow.add_edge("fetch_issues", "format_for_ai")
workflow.add_edge("format_for_ai", "call_gemini")
workflow.add_edge("call_gemini", "parse_and_filter")
workflow.add_edge("parse_and_filter", END) # The final step

# 4. Compile the graph into a runnable object
app = workflow.compile()

# 5. Define your inputs and run the app!
initial_input = {
    "repo_url": "https://github.com/n8n-io/n8n",
    "user_query": "Find first 30 issues related to python and typescript."
}

final_state = app.invoke(initial_input)

# 6. Print the final results
if final_state.get('error'):
    print("\n--- WORKFLOW FAILED ---")
    print(f"Error: {final_state['error']}")
    if final_state.get('ai_response'):
        print("\nRaw AI Response that caused the error:")
        print(final_state['ai_response'])
else:
    print("\n--- WORKFLOW COMPLETED SUCCESSFULLY! ---")
    print("\nFound Matching Issues:")
    for url in final_state['filtered_issues']:
        print(url)

--- 1. FETCHING GITHUB ISSUES ---


/tmp/ipython-input-847158133.py:9: DeprecationWarning: Argument login_or_token is deprecated, please use auth=github.Auth.Token(...) instead
  g = Github(os.environ['GITHUB_ACCESS_TOKEN'])


--- 2. FORMATTING DATA FOR AI ---
--- 3. CALLING GEMINI ---
--- 4. PARSING AND FILTERING RESPONSE ---

--- WORKFLOW COMPLETED SUCCESSFULLY! ---

Found Matching Issues:
https://github.com/n8n-io/n8n/pull/22860
https://github.com/n8n-io/n8n/issues/22798
https://github.com/n8n-io/n8n/pull/22790
https://github.com/n8n-io/n8n/issues/22772
https://github.com/n8n-io/n8n/issues/22758
https://github.com/n8n-io/n8n/issues/22706
https://github.com/n8n-io/n8n/pull/22656
https://github.com/n8n-io/n8n/issues/22639
https://github.com/n8n-io/n8n/pull/22632
https://github.com/n8n-io/n8n/issues/22614
https://github.com/n8n-io/n8n/issues/22581
